# 05 · Approximate nearest neighbors: speed and recall tradeoffs

An exact scan from lesson 04 compares the query with every vector. With millions of fragments, this can be expensive. ANN limits the number of comparisons but may skip the true neighbor. We will learn two mechanisms: **inverted file (IVF)** via simple k-means partitioning and **HNSW** in local, in-memory Chroma. PQ compresses vectors and reduces memory at the cost of quantization error; we won't pretend that our small example measures production gains.

In [ ]:
# Only cell with imports and project path.
import sys
import time
from pathlib import Path

import chromadb
import numpy as np
from sklearn.cluster import KMeans

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.toml").exists())
sys.path.insert(0, str(ROOT / "notebooks" / "rag"))
from lab import embed_texts, section_records

## 1. Gold standard

We normalize the embeddings and save the exact top-5 for several questions. `recall@5` ANN here means the proportion of **five exact neighbors** found by the approximation. This is not yet a measure of the answer to the question.

In [ ]:
records = section_records()
raw = embed_texts([row["text"] for row in records])
unit = raw / np.linalg.norm(raw, axis=1, keepdims=True)
questions = ["zero denominator in percentage forecast error", "November 2011 header billing",
             "credit note and receivables balance", "stockout effect on forecast backtest",
             "quality incident investigation evidence"]
q_raw = embed_texts(questions)
queries = q_raw / np.linalg.norm(q_raw, axis=1, keepdims=True)
exact = [np.argsort(-(unit @ q))[:5] for q in queries]
len(records), [records[int(i)]["id"] for i in exact[0]]

## 2. IVF in a dozen lines

KMeans determines the centroids. After normalization, we assign documents and select query cells based on the same cosine similarity used for document ranking. The query first selects `nprobe` centroids and then compares the query only against their members. A smaller `nprobe` means fewer candidates and a potential drop in recall. This is a didactic equivalent of the IVF idea, not a full implementation of FAISS.

In [ ]:
centroids = KMeans(n_clusters=8, random_state=42, n_init=10).fit(unit)
centroid_unit = centroids.cluster_centers_ / np.linalg.norm(centroids.cluster_centers_, axis=1, keepdims=True)
assignments = np.argmax(unit @ centroid_unit.T, axis=1)
ivf_results = []
for nprobe in (1, 2, 4, 8):
    recalls = []
    examined = []
    for q, gold in zip(queries, exact):
        centroid_scores = centroid_unit @ q
        chosen = np.argsort(-centroid_scores)[:nprobe]
        candidates = np.flatnonzero(np.isin(assignments, chosen))
        ranked = candidates[np.argsort(-(unit[candidates] @ q))[:5]]
        recalls.append(len(set(ranked) & set(gold)) / len(gold))
        examined.append(len(candidates))
    ivf_results.append({"nprobe": nprobe, "average_comparisons": round(float(np.mean(examined)), 1),
                        "recall@5_vs_exact": round(float(np.mean(recalls)), 3)})
ivf_results

## 3. Chroma HNSW in memory

HNSW stores a neighborhood graph and explores a portion of it during search. `max_neighbors` affects the size of the graph, while `ef_search` affects the search depth. Chroma returns **cosine distance** (less = closer), whereas our exact scan returns cosine similarity (more = closer). On a small corpus, Chroma's latency may be higher than a full scan; do not draw conclusions about scaling from these milliseconds.

In [ ]:
client = chromadb.EphemeralClient()
hnsw_results = []
for ef in (10, 100):
    collection = client.create_collection(f"rag-hnsw-ef-{ef}", configuration={"hnsw": {"space": "cosine", "ef_search": ef, "max_neighbors": 16}})
    collection.add(ids=[row["id"] for row in records], embeddings=unit.tolist(),
                   documents=[row["text"] for row in records])
    start = time.perf_counter()
    found = collection.query(query_embeddings=queries.tolist(), n_results=5, include=["distances"])
    elapsed = time.perf_counter() - start
    recalls = [len(set(ids) & {records[int(i)]["id"] for i in gold}) / 5 for ids, gold in zip(found["ids"], exact)]
    hnsw_results.append({"ef_search": ef, "recall@5_vs_exact": round(float(np.mean(recalls)), 3),
                         "time_5_queries_ms": round(1000 * elapsed, 2), "first_distance": round(found["distances"][0][0], 3)})
hnsw_results

## When to use which method?

| Method | Memory Cost | Tuning | Typical Risk |
|---|---|---|---|
| Exact | full vectors | no ANN threshold | full scan time |
| IVF | vectors + centroids | `nlist`, `nprobe` | good neighbor in an unvisited cell |
| HNSW | vectors + graph | `M`, `efConstruction`, `efSearch` | too shallow exploration |
| PQ | compressed codes | number of parts and bits | distance error after quantization |

**Exercise:** change `nprobe` and check the query for which recall dropped. In practice, evaluate simultaneously recall relative to exact, domain relevance, latency, memory, updates, and access filters. [FAISS index guide](https://github.com/facebookresearch/faiss/wiki/Faiss-indexes) · [Chroma HNSW](https://cookbook.chromadb.dev/core/configuration/).